# Thêm Thư Viện

In [16]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [17]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=Library_DWH;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

# Đọc data

## Đọc data từ SQL Server

In [18]:
# Hàm đọc dữ liệu từng phần và xử lý lỗi
def fetch_data_in_batches(query_base, connection, batch_size=100):
    offset = 0
    all_data = []  # Lưu tất cả các hàng hợp lệ
    while True:
        query = f"""
        {query_base}
        ORDER BY ID
        OFFSET {offset} ROWS FETCH NEXT {batch_size} ROWS ONLY
        """
        try:
            # Đọc dữ liệu batch hiện tại
            df_batch = pd.read_sql(query, connection)
            if df_batch.empty:  # Nếu không còn dữ liệu, dừng vòng lặp
                break
            all_data.append(df_batch)  # Lưu batch hợp lệ
            offset += batch_size  # Tăng offset để đọc batch tiếp theo
        except Exception as e:
            print(f"Lỗi xảy ra khi xử lý batch từ {offset}: {e}")
            offset += batch_size  # Bỏ qua batch bị lỗi và tiếp tục
    # Gộp tất cả các batch thành DataFrame duy nhất
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

In [ ]:
query_Xepgia = """
SELECT ID,
       Tai_lieu_ID, 
       Ma_xep_gia,
       Ten_thu_vien_ID,
       Kho_ID,
       Ngay_bo_sung,
       Cho_nhap_kho,
       InUsed,
       Gia,
       Gia_tien,
       InCirculation,
       Kiem_ke,
       dbo.DecodeUTF8String(Nguon_Nhap) AS Nguon_Nhap,
       dbo.DecodeUTF8String(Callnumber) AS Callnumber,
       So_HD
FROM Ma_xep_gia
"""
df_xepgia = fetch_data_in_batches(query_Xepgia, conn_libol, batch_size=100) # Gọi hàm để lấy dữ liệu
print(df_xepgia)

C:\Users\admin\AppData\Local\Temp\ipykernel_29796\3076783788.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_batch = pd.read_sql(query, connection)


        ID  Tai_lieu_ID Ma_xep_gia  Ten_thu_vien_ID  Kho_ID Ngay_bo_sung  \
0   182098           31  SKV000134                1      19   2002-08-23   
1   182099           31  SKV000135                1       5   2002-08-23   
2   182100           31  SKV000136                1       5   2002-08-23   
3   182101           31  SKV000138                1       5   2002-08-23   
4   182102           31  SKV000137                1       5   2002-08-23   
..     ...          ...        ...              ...     ...          ...   
95  182427          191  SKV000082                1       5   2002-08-30   
96  182428          191  SKV000083                1       5   2002-08-30   
97  182429          191  SKV000084                1       5   2002-08-30   
98  182430          191  SKV000085                1       5   2002-08-30   
99  182434          195  SKV000088                1      18   2002-08-30   

    Cho_nhap_kho  InUsed Gia  Gia_tien  InCirculation  Kiem_ke Nguon_Nhap  \
0         

### Tạo dataframe backup 

In [20]:
valid_rows = [] # Danh sách lưu các dòng hợp lệ
for index, row in df_xepgia.iterrows(): # Copy từng dòng
    try:
        valid_rows.append(row.copy()) # Thử copy dòng
    except Exception as e:
        print(f"Lỗi khi copy dòng {index}: {e}")
        continue  # Bỏ qua dòng lỗi và tiếp tục

df_xepgia_backup = pd.DataFrame(valid_rows) # Tạo DataFrame mới từ các dòng hợp lệ
print("Số dòng trong df_xepgia:", len(df_xepgia)) # Kiểm tra số lượng dòng
print("Số dòng trong df_xepgia_backup:", len(df_xepgia_backup)) # Kiểm tra số lượng dòng

Số dòng trong df_xepgia: 100
Số dòng trong df_xepgia_backup: 100


### [Nếu cần] lấy lại data từ backup

In [21]:
valid_rows = [] # Danh sách lưu các dòng hợp lệ
for index, row in df_xepgia_backup.iterrows(): # Copy từng dòng
    try:
        valid_rows.append(row.copy()) # Thử copy dòng
    except Exception as e:
        print(f"Lỗi khi copy dòng {index}: {e}")
        continue  # Bỏ qua dòng lỗi và tiếp tục

df_xepgia = pd.DataFrame(valid_rows) # Tạo DataFrame mới từ các dòng hợp lệ
print("Số dòng trong df_xepgia_backup:", len(df_xepgia_backup)) # Kiểm tra số lượng dòng
print("Số dòng trong df_xepgia:", len(df_xepgia)) # Kiểm tra số lượng dòng

Số dòng trong df_xepgia_backup: 100
Số dòng trong df_xepgia: 100


# Xử lý data

## Thêm 1 dòng giả định none

In [22]:
# Tạo DataFrame `new_row` chứa dòng dữ liệu giả định
new_row = pd.DataFrame({
    'ID': [0],
    'Tai_lieu_ID': ['0'],
    'Ma_xep_gia': ['(Không xác định)'],
    'Ten_thu_vien_ID': ['0'],
    'Kho_ID': [0],
    'Ngay_bo_sung': ['1024-01-01 00:00:00'],
    'Cho_nhap_kho': [0],
    'InUsed': [0],
    'Gia': [0],
    'Gia_tien': [0],
    'InCirculation': [0],
    'Kiem_ke': [0],
    'Nguon_Nhap': ['(Không xác định)'],
    'Callnumber': ['(Không xác định)'],
    'So_HD': ['(Không xác định)']
})
# Thêm dòng dữ liệu giả định vào `df` bằng `pd.concat`
df_xepgia = pd.concat([df_xepgia, new_row], ignore_index=True) # Thêm vào dataframe
df_xepgia = df_xepgia.sort_values(by="ID", ascending=True).reset_index(drop=True) # sắp xếp từ nhỏ đến lớn           
print(df_xepgia)

         ID Tai_lieu_ID        Ma_xep_gia Ten_thu_vien_ID  Kho_ID  \
0         0           0  (Không xác định)               0       0   
1    182098          31         SKV000134               1      19   
2    182099          31         SKV000135               1       5   
3    182100          31         SKV000136               1       5   
4    182101          31         SKV000138               1       5   
..      ...         ...               ...             ...     ...   
96   182427         191         SKV000082               1       5   
97   182428         191         SKV000083               1       5   
98   182429         191         SKV000084               1       5   
99   182430         191         SKV000085               1       5   
100  182434         195         SKV000088               1      18   

            Ngay_bo_sung  Cho_nhap_kho  InUsed Gia  Gia_tien  InCirculation  \
0    1024-01-01 00:00:00             0       0   0       0.0              0   
1    2002-08-

## Xử lý data rỗng hoặc " "

In [23]:
df_xepgia = df_xepgia.replace('', None)
df_xepgia = df_xepgia.replace(np.nan, None)
print(df_xepgia)

         ID Tai_lieu_ID        Ma_xep_gia Ten_thu_vien_ID  Kho_ID  \
0         0           0  (Không xác định)               0       0   
1    182098          31         SKV000134               1      19   
2    182099          31         SKV000135               1       5   
3    182100          31         SKV000136               1       5   
4    182101          31         SKV000138               1       5   
..      ...         ...               ...             ...     ...   
96   182427         191         SKV000082               1       5   
97   182428         191         SKV000083               1       5   
98   182429         191         SKV000084               1       5   
99   182430         191         SKV000085               1       5   
100  182434         195         SKV000088               1      18   

            Ngay_bo_sung  Cho_nhap_kho  InUsed   Gia  Gia_tien  InCirculation  \
0    1024-01-01 00:00:00             0       0     0       0.0              0   
1    2002

## Xử lý kiểu date

In [24]:
query_date = "SELECT Date_key FROM olap.DIM_Date"
df_date = pd.read_sql(query_date, conn_dwh_library)
date_ids = set(df_date['Date_key'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong date_key của bảng DIM_date hay không ?
df_xepgia['Ngay_bo_sung'] = pd.to_datetime(df_xepgia['Ngay_bo_sung'], errors='coerce')
df_xepgia['Ngay_bo_sung'] = df_xepgia['Ngay_bo_sung'].apply(lambda x: int(x.strftime('%Y%m%d')) if pd.notna(x) and int(x.strftime('%Y%m%d')) in date_ids else 0)
print(df_xepgia[['Ngay_bo_sung']])

     Ngay_bo_sung
0               0
1        20020823
2        20020823
3        20020823
4        20020823
..            ...
96       20020830
97       20020830
98       20020830
99       20020830
100      20020830

[101 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_29796\3300255174.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_date = pd.read_sql(query_date, conn_dwh_library)
C:\Users\admin\AppData\Local\Temp\ipykernel_29796\3300255174.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_xepgia['Ngay_bo_sung'] = pd.to_datetime(df_xepgia['Ngay_bo_sung'], errors='coerce')


## Xử lý ID_tai_lieu

In [25]:
query_Tailieu = "SELECT ID_tai_lieu FROM olap.DIM_Tai_lieu"
df_tailieu = pd.read_sql(query_Tailieu, conn_dwh_library)
tailieu_ids = set(df_tailieu['ID_tai_lieu'])
# chuyển date về dang int 
# kiểm tra nhưng ngày đó có tồn tại trong ID_tai_lieu của bảng DIM_Tai_lieu hay không ?
df_xepgia['Tai_lieu_ID'] = df_xepgia['Tai_lieu_ID'].apply(lambda x: x if pd.notna(x) and x in tailieu_ids else 0)
print(df_xepgia[['Tai_lieu_ID']])

     Tai_lieu_ID
0              0
1             31
2             31
3             31
4             31
..           ...
96           191
97           191
98           191
99           191
100          195

[101 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_29796\3432875913.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tailieu = pd.read_sql(query_Tailieu, conn_dwh_library)


## Xử lý ID_thu_vien

In [26]:
# truy xuất dữ liệu từ libol ra
query_Thuvien = "SELECT Thu_vien_ID, dbo.DecodeUTF8String(Ten_viet_tat) AS Ten_viet_tat FROM Thu_vien"
df_thuvien = pd.read_sql(query_Thuvien, conn_libol)
map_dict = dict(zip(df_thuvien['Thu_vien_ID'], df_thuvien['Ten_viet_tat'])) # mapping lại 1:DHSPKT 2:ĐHSPKT
# kiểm tra id trong mapping nếu trùng với Thu_vien_ID thì đổi thành Ten_viet_tat
# vd 1 -> DHSPKT
df_xepgia['Ten_thu_vien_ID'] = df_xepgia['Ten_thu_vien_ID'].map(map_dict).fillna(df_xepgia['Ten_thu_vien_ID'])
# truy xuất dữ liệu từ dwh bảng DIM_Thu_vien ra
query_Thuvien = "SELECT ID_thu_vien FROM olap.DIM_Thu_vien"
df_thuvien = pd.read_sql(query_Thuvien, conn_dwh_library)
thuvien_ids = set(df_thuvien['ID_thu_vien'])
# kiểm tra tên viết tắt ở trên kìa có trong bảng DIM_Thu_vien không? có thì bỏ qua không thì bằng 0
df_xepgia['Ten_thu_vien_ID'] = df_xepgia['Ten_thu_vien_ID'].apply(lambda x: x if x in thuvien_ids else 0)
print(df_xepgia[['Ten_thu_vien_ID']])

    Ten_thu_vien_ID
0                 0
1            DHSPKT
2            DHSPKT
3            DHSPKT
4            DHSPKT
..              ...
96           DHSPKT
97           DHSPKT
98           DHSPKT
99           DHSPKT
100          DHSPKT

[101 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_29796\2339722395.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_thuvien = pd.read_sql(query_Thuvien, conn_libol)
C:\Users\admin\AppData\Local\Temp\ipykernel_29796\2339722395.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_thuvien = pd.read_sql(query_Thuvien, conn_dwh_library)


## xử lý ID_kho

In [27]:
query_Kho = "SELECT ID_kho FROM olap.DIM_Kho"
df_kho = pd.read_sql(query_Kho, conn_dwh_library)
kho_ids = set(df_kho['ID_kho'])
# chuyển date về dang int 
# kiểm tra id đó có tồn tại trong ID_kho của bảng DIM_Kho hay không ?
df_xepgia['Kho_ID'] = df_xepgia['Kho_ID'].apply(lambda x: x if pd.notna(x) and x in kho_ids else 0)
print(df_xepgia[['Kho_ID']])

     Kho_ID
0         0
1        19
2         5
3         5
4         5
..      ...
96        5
97        5
98        5
99        5
100      18

[101 rows x 1 columns]


C:\Users\admin\AppData\Local\Temp\ipykernel_29796\933683982.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_kho = pd.read_sql(query_Kho, conn_dwh_library)


## Xử lý Gia_tien

In [28]:
# Gia với Gia_tien giống nhau
# Gia kiểu varchar và Gia_tien kiểu money
# quét qua từng hàng
for index, row in df_xepgia.iterrows(): 
    if pd.isnull(row['Gia_tien']): # nếu gia tiền null và gia đang có giá trị thì chuyển về float và dán lại
        if not pd.isnull(row['Gia']):
            df_xepgia.at[index, 'Gia_tien'] = float(row['Gia'])
        else:
            df_xepgia.at[index, 'Gia_tien'] = 0;
print(df_xepgia['Gia_tien'])

0      0.0
1      0.0
2      0.0
3      0.0
4      0.0
      ... 
96     0.0
97     0.0
98     0.0
99     0.0
100    0.0
Name: Gia_tien, Length: 101, dtype: float64


## Load data

### [Nếu cần] Clear bảng

In [29]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Xep_gia"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [30]:
# Tạo cursor để thao tác với cơ sở dữ liệu
cursor_dwh = conn_dwh_library.cursor()

# Chuẩn bị câu lệnh chèn dữ liệu
insert_query = """
                INSERT INTO olap.DIM_Xep_gia (
                    ID_xep_gia, ID_tai_lieu, Ma_xep_gia,
                    ID_thu_vien, ID_kho, 
                    Ngay_bo_sung,
                    Cho_nhap_kho, InUsed,
                    Gia_tien, InCirculation, Kiem_ke,
                    Nguon_Nhap, Callnumber, So_HD
                ) 
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
               """
# Chuyển đổi dữ liệu từ DataFrame thành danh sách các tuple để chèn
data_to_insert = [
    (
        row['ID'], row['Tai_lieu_ID'], row['Ma_xep_gia'], 
        row['Ten_thu_vien_ID'], row['Kho_ID'], 
        row['Ngay_bo_sung'],
        row['Cho_nhap_kho'], row['InUsed'],
        row['Gia_tien'], row['InCirculation'], row['Kiem_ke'], 
        row['Nguon_Nhap'], row['Callnumber'], row['So_HD']
    )
    for index, row in df_xepgia.iterrows()
]
# Sử dụng executemany để chèn dữ liệu cùng lúc
cursor_dwh.executemany(insert_query, data_to_insert)
# Commit thay đổi
conn_dwh_library.commit()
# Đóng cursor và kết nối
cursor_dwh.close()
conn_dwh_library.close()
